In [19]:
from sqlalchemy import create_engine
import pandas as pd

# Replace with your actual path to the .sqlite file
engine = create_engine('sqlite:///D:\Others\Movie Recommender\movie_db.sqlite')

# Test the connection: read a table
df = pd.read_sql('SELECT * FROM Movies m', engine)
print(df.head())


   movie_id          title  release_year  duration  \
0         1     The Matrix          1999       136   
1         2      Inception          2010       148   
2         3   Interstellar          2014       169   
3         4  The Godfather          1972       175   
4         5     Fight Club          1999       139   

                                         description  
0      A hacker discovers the nature of his reality.  
1             A thief steals secrets through dreams.  
2      Explorers travel through a wormhole in space.  
3  The aging patriarch of an organized crime dyna...  
4  An insomniac office worker and a devil-may-car...  


In [20]:
from sqlalchemy import create_engine
import pandas as pd
engine = create_engine('sqlite:///D:\Others\Movie Recommender\movie_db.sqlite')
# Load necessary tables
ratings = pd.read_sql('SELECT * FROM Ratings', engine)
movies = pd.read_sql('SELECT * FROM Movies', engine)
genres = pd.read_sql('SELECT * FROM Genres', engine)
movie_genres = pd.read_sql('SELECT * FROM MovieGenres', engine)

# Merge for enriched movie data
movie_data = movies.merge(movie_genres, on='movie_id').merge(genres, on='genre_id')
full_data = ratings.merge(movie_data, on='movie_id')

In [21]:
user_id = 1  # You can change this to any user_id from your Users table

# Get this user's rated movies
user_ratings = full_data[full_data['user_id'] == user_id]

# Get user's favorite genres
fav_genres = (
    user_ratings.groupby('genre_name')['rating']
    .mean()
    .sort_values(ascending=False)
)

print("User's favorite genres:\n", fav_genres)


User's favorite genres:
 genre_name
Action    4.5
Sci-Fi    4.5
Name: rating, dtype: float64


In [22]:
top_genre = fav_genres.index[0]  # pick the top genre

# Movies the user hasn’t rated yet
unrated = full_data[~full_data['movie_id'].isin(user_ratings['movie_id'])]

# Filter for top genre
recommend_pool = unrated[unrated['genre_name'] == top_genre]

# Recommend top-rated movies from this genre
recommendations = (
    recommend_pool.groupby(['movie_id', 'title'])['rating']
    .mean()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

print(f"\n🎬 Top movie recommendations for User {user_id} based on genre '{top_genre}':")
print(recommendations)



🎬 Top movie recommendations for User 1 based on genre 'Action':
Empty DataFrame
Columns: [movie_id, title, rating]
Index: []


In [23]:
user_id = 1
user_ratings = full_data[full_data['user_id'] == user_id]

print(user_ratings[['title', 'genre_name', 'rating']])


        title genre_name  rating
0  The Matrix     Sci-Fi     5.0
1  The Matrix     Action     5.0
6   Inception     Sci-Fi     4.0
7   Inception     Action     4.0


In [24]:
action_movies = full_data[full_data['genre_name'] == 'Action']
print(action_movies[['title']].drop_duplicates())


        title
1  The Matrix
3   Inception


In [25]:
seen_movie_ids = user_ratings['movie_id'].unique()
unseen_actions = action_movies[~action_movies['movie_id'].isin(seen_movie_ids)]

print(unseen_actions[['title', 'rating']].dropna().sort_values('rating', ascending=False).head())


Empty DataFrame
Columns: [title, rating]
Index: []
